# Chapter 1: Proximity Service
*System Design Interview Volume 2*

## TL;DR

A proximity service finds nearby places (restaurants, hotels, gas stations) given a user's location and search radius. The core challenge is **geospatial indexing** -- mapping 2D coordinates to an efficiently searchable 1D structure. **Geohash** (hash-based) and **Quadtree** (tree-based) are the two most practical approaches. The system is read-heavy and stateless, making it easy to scale horizontally with replicas and caching.

## Requirements

| Requirement | Detail |
|---|---|
| Nearby search | Return businesses within a given lat/lng + radius |
| Business CRUD | Add, delete, update businesses (next-day effective) |
| Business detail | View full business information |
| Low latency | Sub-second search responses |
| High availability | Handle peak traffic in dense areas |
| Data privacy | GDPR / CCPA compliance |

## Estimation: Search QPS and Geohash Memory

In [ ]:
# Back-of-the-envelope: Proximity Service
dau = 100_000_000          # 100M daily active users
businesses = 200_000_000   # 200M businesses
searches_per_user = 5      # queries per user per day
seconds_per_day = 86_400   # ~10^5

search_qps = dau * searches_per_user / seconds_per_day
print(f"Search QPS: {search_qps:,.0f}")

# Geohash Redis cache memory (3 precisions: lengths 4, 5, 6)
bytes_per_id = 8
precisions = 3
redis_memory_gb = bytes_per_id * businesses * precisions / (1024**3)
print(f"Redis cache for geohash index: {redis_memory_gb:.1f} GB")

# Quadtree memory
leaf_nodes = businesses / 100  # max 100 per leaf
internal_nodes = leaf_nodes / 3
leaf_bytes = 832  # 32 coords + 800 biz IDs
internal_bytes = 64  # 32 coords + 32 pointers
qt_memory_gb = (leaf_nodes * leaf_bytes + internal_nodes * internal_bytes) / (1024**3)
print(f"Quadtree memory: {qt_memory_gb:.2f} GB")

## High-Level Design

```mermaid
graph TB
    Client[Mobile / Web Client]
    LB[Load Balancer]
    LBS[Location-Based Service<br>stateless, read-only]
    BIZ[Business Service<br>CRUD operations]
    RC1[Redis Cache<br>Geohash Index]
    RC2[Redis Cache<br>Business Info]
    R1[(Read Replica 1)]
    R2[(Read Replica 2)]
    P[(Primary DB)]

    Client --> LB
    LB -->|GET /v1/search/nearby| LBS
    LB -->|/v1/businesses| BIZ
    LBS --> RC1
    LBS --> RC2
    LBS --> R1
    LBS --> R2
    BIZ --> P
    P -->|Replicate| R1
    P -->|Replicate| R2

    style LBS fill:#b3e6b3,stroke:#333
    style BIZ fill:#ffcc99,stroke:#333
    style RC1 fill:#ff9999,stroke:#333
    style RC2 fill:#ff9999,stroke:#333
```

**Key design decisions:**
- **Separate read/write paths** -- LBS handles search (high QPS, read-only); Business Service handles CRUD (low QPS, writes)
- **Stateless LBS** -- easy horizontal scaling
- **Database replicas** -- serve read-heavy search traffic
- **Redis caching** -- geohash-to-business-IDs + business objects

## Deep Dive: Geohash Encoding

Geohash recursively bisects the world into quadrants, encoding each split as a bit. The result is a base-32 string where **longer shared prefixes mean closer proximity**.

| Geohash Length | Grid Size | Use Case |
|---|---|---|
| 4 | 39.1km x 19.5km | 20km radius search |
| 5 | 4.9km x 4.9km | 1-2km radius search |
| 6 | 1.2km x 609m | 0.5km radius search |

### Boundary Issues and Solutions

1. **No shared prefix** -- Two nearby points can cross the equator/prime meridian and share no prefix. Solution: always query the **current grid + 8 neighbors**.
2. **Shared prefix, different grid** -- Two close points may fall in adjacent geohash cells. Same solution: query neighbors.

### Search Algorithm

1. Compute geohash at the appropriate length for the search radius
2. Calculate 8 neighboring geohash cells
3. Query Redis for business IDs in all 9 cells (parallel)
4. Hydrate business objects from Business Info cache
5. Filter by distance, rank, and return

## Deep Dive: Quadtree vs Geohash

```mermaid
graph TB
    subgraph QT["Quadtree (adaptive subdivision)"]
        QR[Root: World] --> Q1[NW]
        QR --> Q2[NE]
        QR --> Q3[SW]
        QR --> Q4[SE]
        Q1 --> L1["Leaf: 87 biz"]
        Q1 --> L2["Leaf: 45 biz"]
    end

    subgraph GH["Geohash (fixed grid)"]
        G1["9q8zn1"] --- G2["9q8zn2"]
        G2 --- G3["9q8zn3"]
        G3 --- G4["9q8zn4"]
    end
```

| Feature | Geohash | Quadtree |
|---|---|---|
| Implementation | Simple, SQL-friendly | In-memory tree, more complex |
| Grid size | Fixed per precision level | Adaptive to data density |
| K-nearest query | Requires expanding search | Natural fit (subdivides by count) |
| Update cost | O(1) row insert/delete | O(log n) tree traversal + locking |
| Memory | ~5 GB in Redis | ~1.7 GB in-memory |
| Used by | Redis, MongoDB, Lyft, Bing | Yext |

**Recommendation:** Use geohash for simplicity; quadtree for k-nearest or density-adaptive needs.

## Trade-offs

| Decision | Pro | Con |
|---|---|---|
| Geohash over Quadtree | Simple, easy to cache and update | Fixed grid, not density-adaptive |
| Read replicas over sharding | Simpler ops, data fits one server | Limited write scalability |
| Redis caching | Sub-ms lookups, precomputed per radius | Cache invalidation on business updates |
| Next-day business updates | Simplifies cache invalidation via nightly job | Stale data for up to 24h |
| Multi-region deployment | Low latency, privacy compliance | Data sync complexity |

## Takeaways

1. **Geospatial indexing** is the core challenge -- geohash and quadtree are the two most interview-relevant approaches
2. **Read-heavy workloads** benefit from stateless services + read replicas + aggressive caching
3. **Always query neighbors** when using geohash to handle boundary issues
4. **Google S2** offers flexible cell sizes via Hilbert curves but is complex to explain in interviews
5. **Cache at multiple precisions** (geohash lengths 4, 5, 6) to serve different search radii efficiently

## Related Concepts

- [[geospatial-indexing]] -- Core indexing techniques for spatial data
- [[geohash]] -- Hash-based spatial encoding
- [[quadtree]] -- Tree-based adaptive spatial partitioning
- [[geospatial-caching]] -- Redis caching patterns for proximity queries
- [[google-s2]] -- Hilbert-curve based spherical indexing